### Load Packages

In [1]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import LineString, Point
from bus_route_plot import original_route, new_route, original_and_new_route
from mrt_map import get_mrt_map


### Data Pre-Processing (Weekdays, Peak Hour)

In [2]:
df_202407 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202407.csv")
df_202408 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202408.csv")
df_202409 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202409.csv")
combined_df = pd.concat([df_202407, df_202408, df_202409], ignore_index=True)

# Filter for DAY_TYPE == 'WEEKDAY' and TIME_PER_HOUR for peak hours [7, 8, 9, 10, 17, 18, 19, 20]
filtered_df = combined_df[(combined_df['DAY_TYPE'] == 'WEEKENDS/HOLIDAY') & 
                          (combined_df['TIME_PER_HOUR'].isin([9, 10, 11, 12, 17, 18, 19, 20, 21]))]
summarised_df = pd.DataFrame(columns=['PT_CODE', 'TOTAL_VOLUME'])
# Group by 'PT_CODE' and calculate the sum of tap-in and tap-out volumes
grouped = filtered_df.groupby('PT_CODE').agg(
    TOTAL_TAP_IN_VOLUME=('TOTAL_TAP_IN_VOLUME', 'sum'),
    TOTAL_TAP_OUT_VOLUME=('TOTAL_TAP_OUT_VOLUME', 'sum')).reset_index()
# Create a new column for the total volume (sum of tap-in and tap-out volumes)
grouped['TOTAL_VOLUME'] = grouped['TOTAL_TAP_IN_VOLUME'] + grouped['TOTAL_TAP_OUT_VOLUME']
summarised_df['PT_CODE'] = grouped['PT_CODE']
summarised_df['TOTAL_VOLUME'] = grouped['TOTAL_VOLUME']
summarised_df = summarised_df.sort_values(by='TOTAL_VOLUME', ascending=False)
trunkroutes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")
trunkroutes_grouped = trunkroutes.groupby('BusStopCode').first().reset_index()
# Merge summarised_df with trunkroutes based on PT_CODE == BusStopCode
merged_df = pd.merge(summarised_df, trunkroutes_grouped[['BusStopCode', 'Description', 'Latitude', 'Longitude','Direction']], 
                     left_on='PT_CODE', right_on='BusStopCode', how='left')
merged_df = merged_df.drop(columns=['BusStopCode'])
merged_df.to_csv('location_popular_bus_Stops.csv', index=False)
filtered_df = merged_df[~merged_df['Description'].str.contains('Stn|Int', na=False)].reset_index().drop('index', axis=1)

### Retrieving MRT Map

In [3]:
from mrt_map import get_mrt_map

singapore = get_mrt_map()
singapore

/Users/lilyrozana/Documents/GitHub/DSA4264/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()


### Plotting popular bus stops across SG on weekdays, during peak hour

In [5]:
def scale_marker_size(volume, min_size=5, max_size=15):
    volume_range = merged_df['TOTAL_VOLUME'].max() - merged_df['TOTAL_VOLUME'].min()
    if volume_range == 0:
        return min_size  # Avoid division by zero
    scaled_size = ((volume - merged_df['TOTAL_VOLUME'].min()) / volume_range) * (max_size - min_size) + min_size
    return scaled_size

# Plot first 100 rows on the singapore_mrt map
for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(singapore)
singapore

## Proposed Route 1 & 2
#### Check if selected bus stops are currently in any bus service route

In [6]:
route1a= ['17041', '17159', '12091', '43181', '43419', '28461', '28491', '28511', '28401', '21441']
route1a_proposed = ['17171', '17041', '17159', '12091', '43181', '43419', 
                    '28461', '28491', '28511', '28401','21431', '22009']

route1b = ['63241', '63059', '64419', '64119', '64111', '66331', '66339', '66271', 
        '54481', '54489', '54248', '54241', '53389', '53381', '52059', '52361', 
        '60089', '60179', '60161', '80071', '80069', '80089']
route1b_proposed = ['64009', '63249', '63059', '64419', '64119', '66339', 
                    '54489', '54248', '53389', '52059', '52361', '60081', 
                    '60179', '80071', '80089', '80069', '80009']

In [7]:
trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

# Retrieving the bus stop code from the bus stop label
busstop1 = trunk_bus_routes[trunk_bus_routes['Description'].str.contains('Keming Pr Sch', case=False, na=False)]['BusStopCode'].unique()
busstop2 = trunk_bus_routes[trunk_bus_routes['Description'].str.contains('Aft Bt Batok Stn/Blk 628', case=False, na=False)]['BusStopCode'].unique()

# Function for checking proportion of bus stops in proposed routes that are present in any bus service
trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
def calculate_proportion(group, list_of_buses):
    matching_stops = group['BusStopCode'].isin(list_of_buses).sum()
    proportion = matching_stops / len(group)
    return proportion

# Check if there is an existing bus service that goes through the bus stops for route1a
grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1a)).reset_index(name='Proportion')
grouped_sorted = grouped.sort_values(by='Proportion', ascending=False)
grouped_sorted

# Plot that bus service on the map
# original_route(grouped_sorted['ServiceNo'].iloc[0])

# Check if there is an existing bus service that goes through the bus stops for route1b
grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1b)).reset_index(name='Proportion')
grouped_sorted = grouped.sort_values(by='Proportion', ascending=False)
grouped_sorted

/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_3410/3402257573.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1a)).reset_index(name='Proportion')
/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_3410/3402257573.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gr

,ServiceNo,Direction,Proportion
231,21A,1,0.250000
128,159B,1,0.200000
364,72B,1,0.200000
88,13A,1,0.166667
32,115,1,0.153846
...,...,...,...
488,97,1,0.000000
487,96B,1,0.000000
486,96A,1,0.000000
485,969A,1,0.000000


### Proposed Route 1: West Rush Hour Express Service - for Weekday Peak Hour 
### Proposed Route 2: East Rush Hour Express Service - for Weekday Peak Hour 

In [8]:
# Plotting Proposed Route 1
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(route1a_proposed)]
filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)  
filtered_stops = filtered_stops.set_index('BusStopCode').loc[route1a_proposed].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(singapore)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(singapore)

# Plotting Proposed Route 2
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(route1b_proposed)]
filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)  
filtered_stops = filtered_stops.set_index('BusStopCode').loc[route1b_proposed].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(singapore)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(singapore)

singapore

/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_3410/3551002502.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)
/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_3410/3551002502.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)


## Proposed Route 3: BTO Express Service - Weekday Peak Hour

### Display 2024 BTO Projects on Map

In [9]:
singapore = get_mrt_map()
## BTOs with over 1,000 units
taman_jurong_skyline = [1.3270289195127982, 103.72582148080623]
tanjong_rhu = [1.2996481809089269, 103.88025809169478]
teban_breeze = [1.3218378113117057, 103.74476401645141]
chencharu_hills = [1.419134809710128, 103.82527849062635]
marsiling_peak = [1.444461323034268, 103.77610683781455]
woodgrove_edge = [1.4290005301535544, 103.78480587301898]

folium.CircleMarker(
    location=taman_jurong_skyline,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Taman Jurong Skyline", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=tanjong_rhu,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Tanjong Rhu Riverfront I & II", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=teban_breeze,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Teban Breeze", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=chencharu_hills,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Chenchura Hills", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=marsiling_peak,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Marsiling Peak", max_width=100)
).add_to(singapore)

folium.CircleMarker(
    location=woodgrove_edge,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("WoodGrove Edge", max_width=100)
).add_to(singapore)


# Display the map
singapore

/Users/lilyrozana/Documents/GitHub/DSA4264/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()


### Plot Routes for Buses Servicing Each BTO

In [10]:
trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

nearest_stop_codes = {
    'taman_jurong_skyline': 21769,
    'tanjong_rhu': 90051,
    'teban_breeze': 20261,
    'chencharu_hills': 57069,
    'marsiling_peak': 46121,
    'woodgrove_edge': 46229
}

bto_colors = {
    'taman_jurong_skyline': 'orange',
    'tanjong_rhu': 'green',
    'teban_breeze': 'red',
    'chencharu_hills': 'purple',
    'marsiling_peak': 'brown',
    'woodgrove_edge': 'pink'
}

filtered_routes = {}

for bto, stop_code in nearest_stop_codes.items():
    # Find the bus services stopping at the nearest bus stop
    bto_services = trunk_bus_routes[trunk_bus_routes['BusStopCode'] == stop_code]['ServiceNo'].unique()
    
    # Filter the bus_routes DataFrame to include only those services
    filtered_routes[bto] = trunk_bus_routes[trunk_bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service >> add only the markers for the stops they visit
for bto, routes in filtered_routes.items():
    color = bto_colors[bto]  
    for service_no in routes['ServiceNo'].unique():
        # Extract route points for each service
        route_points = routes[routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
        
        folium.PolyLine(
            locations=route_points,
            color=color,  
            weight=3,
            opacity=0.6,
            popup=f"Service {service_no} - {bto}"
        ).add_to(singapore)
        

        for idx, row in routes[routes['ServiceNo'] == service_no].iterrows():
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
            ).add_to(singapore)

for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(singapore)

singapore



### Tanjong Rhu 
* largest BTO project announced in 2024 with 2,063 units
* nearest bus stop only has services 158 and 158B, both to non-central regions

In [11]:
tanjong_rhu_map = get_mrt_map()

tanjong_rhu = [1.2996481809089269, 103.88025809169478]

folium.CircleMarker(
    location=tanjong_rhu,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Tanjong Rhu Riverfront I & II", max_width=100)
).add_to(tanjong_rhu_map)


trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")


nearest_stop_code = 90051  

bto_color = 'green'


bto_services = trunk_bus_routes[trunk_bus_routes['BusStopCode'] == nearest_stop_code]['ServiceNo'].unique()


filtered_routes = trunk_bus_routes[trunk_bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service, and add only the markers for the stops they visit
for service_no in filtered_routes['ServiceNo'].unique():
    # Extract route points for each service
    route_points = filtered_routes[filtered_routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
    
    # Draw the route on the map
    folium.PolyLine(
        locations=route_points,
        color=bto_color,  # Use the color for Tanjong Rhu
        weight=3,
        opacity=0.6,
        popup=f"Service {service_no} - Tanjong Rhu"
    ).add_to(tanjong_rhu_map)
    
    # Add markers for each stop along this route
    for idx, row in filtered_routes[filtered_routes['ServiceNo'] == service_no].iterrows():
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=5,
            color=bto_color,
            fill=True,
            fill_color=bto_color,
            fill_opacity=0.7,
            popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
        ).add_to(tanjong_rhu_map)

for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(tanjong_rhu_map)
tanjong_rhu_map

# Display the map with Tanjong Rhu routes and stops
tanjong_rhu_map


/Users/lilyrozana/Documents/GitHub/DSA4264/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()


## Route 3: BTO Express Route [Tanjong Rhu Riverfront]

In [12]:
tanjong_rhu_riverfront_route = ['90061', '90051', '1039', '7518', '7419', '7319','7111', '7031', '40011', '9037', '9219','9179', '9022','9037','5039','3031']
trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
def calculate_proportion(group, list_of_buses):
    matching_stops = group['BusStopCode'].isin(list_of_buses).sum()
    proportion = matching_stops / len(group)
    return proportion

# Check if there is an existing bus service that goes through the bus stops for stops in tanjong_rhu_riverfront_route
grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, tanjong_rhu_riverfront_route)).reset_index(name='Proportion')
grouped_sorted = grouped.sort_values(by='Proportion', ascending=False)
grouped_sorted

/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_3410/986446605.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, tanjong_rhu_riverfront_route)).reset_index(name='Proportion')


,ServiceNo,Direction,Proportion
414,857B,1,0.096774
204,190A,1,0.095238
234,23,1,0.093023
202,190,1,0.090909
105,147A,1,0.085106
...,...,...,...
527,992,1,0.000000
528,992,2,0.000000
529,993,1,0.000000
530,9A,1,0.000000


In [13]:
trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(tanjong_rhu_riverfront_route)]
filtered_stops = filtered_stops.set_index('BusStopCode').loc[tanjong_rhu_riverfront_route].reset_index()


for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(tanjong_rhu_map)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(tanjong_rhu_map)


bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(tanjong_rhu_map)
tanjong_rhu_map


### Saving Proposed Routes to csv

In [ ]:
route1a_proposed = ['17171', '17041', '17159', '12091', '43181', '43419', 
                    '28461', '28491', '28511', '28401','21431', '22009']

route1b_proposed = ['64009', '63249', '63059', '64419', '64119', '66339', 
                    '54489', '54248', '53389', '52059', '52361', '60081', 
                    '60179', '80071', '80089', '80069', '80009']

tanjong_rhu_riverfront_route = ['90061', '90051', '1039', '7518', '7419', '7319','7111', 
                                '7031', '40011', '9037', '9219','9179', '9022','9037','5039','3031']

In [29]:
trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)

route_data = []

# Function to add route data to the list
def add_route_data(service_name, route):
    for sequence, bus_stop_code in enumerate(route, start=1):
        stop_info = trunk_bus_routes[trunk_bus_routes['BusStopCode'] == bus_stop_code]

        if not stop_info.empty:
            latitude = stop_info['Latitude'].values[0]
            longitude = stop_info['Longitude'].values[0]

            route_data.append({
                "ServiceName": service_name,
                "BusStopCode": bus_stop_code,
                "Stop Sequence": sequence,
                "Latitude": latitude,
                "Longitude": longitude
            })

# Add Tanjong Rhu Riverfront Route
add_route_data("Proposed BTO Route", tanjong_rhu_riverfront_route)

# Add West Rush Hour Express
add_route_data("West Rush Hour Express", route1a_proposed)

# Add East Rush Hour Express
add_route_data("East Rush Hour Express", route1b_proposed)

proposed_routes_df = pd.DataFrame(route_data)
proposed_routes_df.to_csv("Bus_RoutesStopsServices/proposed_bus_route.csv", index=False)

print(proposed_routes_df)

               ServiceName BusStopCode  Stop Sequence  Latitude   Longitude
0       Proposed BTO Route       90061              1  1.298123  103.882409
1       Proposed BTO Route       90051              2  1.297659  103.879173
2       Proposed BTO Route        1039              3  1.298208  103.855491
3       Proposed BTO Route        7518              4  1.300958  103.852112
4       Proposed BTO Route        7419              5  1.306530  103.856163
5       Proposed BTO Route        7319              6  1.310388  103.858707
6       Proposed BTO Route        7111              7  1.308794  103.853175
7       Proposed BTO Route        7031              8  1.305721  103.851206
8       Proposed BTO Route       40011              9  1.306064  103.849088
9       Proposed BTO Route        9037             10  1.302340  103.837013
10      Proposed BTO Route        9219             11  1.307320  103.833161
11      Proposed BTO Route        9179             12  1.305880  103.830490
12      Prop